# HOG Explorer — Lab 012

Interactive notebook for querying HOG hierarchy and visualizing sub-HOG structure
in ESM2 embedding space. Focuses on **HOG 801468** (largest root HOG, 2236 proteins,
depth up to 14).

**Key question:** Can we find a HOG level with well-balanced sub-HOG classes for
meaningful clustering evaluation?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

PROJECT = Path("/home/acefsan/src/dl_bio")
DATASETS = PROJECT / "assets/proteins/datasets"
ANNOTATIONS = PROJECT / "chapters/chapter2/runs/esm2_3b_20260304_015127/dataset/cafa3_annotations.feather"

EMBEDDINGS = {
    "150M": DATASETS / "esm2_150m_embeddings.feather",
    "650M": DATASETS / "all_species_embeddings.feather",
    "3B":   DATASETS / "esm2_3b_embeddings.feather",
}

# Load annotations (deduplicated to unique proteins)
ann = pd.read_feather(ANNOTATIONS)
prot = ann[["EntryID", "Length", "taxonomyID", "scientific_name",
            "hog_id", "roothog_id"]].drop_duplicates("EntryID").reset_index(drop=True)
prot["depth"] = prot["hog_id"].apply(lambda x: len(x.split(".")) - 1 if isinstance(x, str) else -1)

print(f"Proteins: {len(prot):,}")
print(f"  with hog_id: {prot.hog_id.notna().sum():,}")
print(f"  with roothog_id != 0: {(prot.roothog_id != 0).sum():,}")
print(f"  unique root HOGs: {prot[prot.roothog_id != 0].roothog_id.nunique():,}")

## Utility functions

In [ ]:
def trunc_hog(hog_id: str, level: int) -> str:
    """Truncate a HOG ID to a given depth level.
    
    HOG IDs look like: HOG:E0801468.10nwj.6661i.5272b.3915a
    Level 0 = root (HOG:E0801468), level 1 = first split, etc.
    If the protein's depth < level, returns its full hog_id (leaf).
    """
    parts = hog_id.split(".")
    return ".".join(parts[: level + 1]) if len(parts) > level else hog_id


def get_hog_proteins(roothog_id: float, min_depth: int = 0) -> pd.DataFrame:
    """Get all proteins in a root HOG, optionally filtered by minimum depth."""
    mask = (prot["roothog_id"] == roothog_id) & prot["hog_id"].notna()
    if min_depth > 0:
        mask &= prot["depth"] >= min_depth
    return prot[mask].copy()


def hog_level_stats(roothog_id: float, level: int, min_class_size: int = 5,
                    min_depth: int = 0) -> pd.DataFrame:
    """Compute sub-HOG class statistics at a given level within a root HOG.
    
    Args:
        roothog_id: The root HOG to analyze
        level: Truncation level for sub-HOG labels
        min_class_size: Minimum proteins per class to include
        min_depth: Only include proteins with depth >= this value
    
    Returns:
        DataFrame with columns: sub_hog, short_label, count
    """
    h = get_hog_proteins(roothog_id, min_depth=min_depth)
    h["sub_hog"] = h["hog_id"].apply(lambda x: trunc_hog(x, level))
    counts = h["sub_hog"].value_counts().reset_index()
    counts.columns = ["sub_hog", "count"]
    counts["short_label"] = counts["sub_hog"].apply(lambda x: x.split(".")[-1])
    big = counts[counts["count"] >= min_class_size].reset_index(drop=True)
    return big


def balance_report(roothog_id: float, levels: list[int] = None,
                   min_class_size: int = 5, min_depth: int = 0):
    """Print a balance report across multiple levels for a root HOG."""
    if levels is None:
        levels = list(range(1, 8))
    
    h = get_hog_proteins(roothog_id, min_depth=min_depth)
    n_total = len(h)
    
    print(f"Root HOG {int(roothog_id)}: {n_total} proteins"
          + (f" (depth >= {min_depth})" if min_depth else ""))
    print(f"{'Level':<7} {'Classes':>8} {'Coverage':>10} {'Min':>5} {'Max':>5} "
          f"{'Mean':>6} {'Median':>7} {'CV':>6}")
    print("-" * 60)
    
    for level in levels:
        stats = hog_level_stats(roothog_id, level, min_class_size, min_depth)
        if len(stats) < 2:
            print(f"L{level:<6} {'<2 classes':>8}")
            continue
        sizes = stats["count"].values
        cv = np.std(sizes) / np.mean(sizes)
        cov = sizes.sum()
        print(f"L{level:<6} {len(stats):>8} {cov:>6}/{n_total:<4} "
              f"{sizes.min():>5} {sizes.max():>5} {sizes.mean():>6.1f} "
              f"{np.median(sizes):>7.0f} {cv:>6.2f}")


def load_embeddings(model: str, entry_ids: set) -> np.ndarray:
    """Load embeddings for a set of proteins. Returns (aligned_ids, X)."""
    df = pd.read_feather(EMBEDDINGS[model])
    df = df[df["EntryID"].isin(entry_ids)]
    ecols = [c for c in df.columns if c.startswith("ME:")]
    return df["EntryID"].values, df[ecols].values


print("Utilities loaded.")

## Top root HOGs overview

Which root HOGs have the most proteins and deepest hierarchies?

In [ ]:
with_hog = prot[prot["hog_id"].notna()].copy()
summary = (with_hog.groupby("roothog_id")
           .agg(n_proteins=("EntryID", "count"),
                max_depth=("depth", "max"),
                mean_depth=("depth", "mean"),
                n_deep5=("depth", lambda x: (x >= 5).sum()))
           .sort_values("n_proteins", ascending=False)
           .head(20))
summary["roothog_id"] = summary.index.astype(int)
summary = summary[["roothog_id", "n_proteins", "max_depth", "mean_depth", "n_deep5"]]
summary["mean_depth"] = summary["mean_depth"].round(1)
display(summary.reset_index(drop=True))

## HOG 801468 — Balance report

This is the largest root HOG (2236 proteins, max depth 14). Let's see how balance
looks at each level, both for all proteins and for only deep (depth >= 5) proteins.

In [ ]:
ROOTHOG = 801468

print("=== All proteins ===")
balance_report(ROOTHOG, levels=list(range(1, 8)), min_class_size=5)
print()
print("=== Depth >= 3 only ===")
balance_report(ROOTHOG, levels=list(range(1, 8)), min_class_size=5, min_depth=3)
print()
print("=== Depth >= 5 only ===")
balance_report(ROOTHOG, levels=list(range(1, 8)), min_class_size=5, min_depth=5)

## Class size distribution at each level

Visualize how balanced the sub-HOG classes are.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, level in enumerate(range(1, 7)):
    ax = axes[i // 3, i % 3]
    stats = hog_level_stats(ROOTHOG, level, min_class_size=3)
    if len(stats) == 0:
        ax.set_title(f"Level {level}: no classes")
        continue
    
    sizes = stats["count"].sort_values(ascending=False).values
    ax.bar(range(len(sizes)), sizes, color="steelblue", alpha=0.7)
    ax.set_title(f"Level {level}: {len(sizes)} classes (>=3)")
    ax.set_xlabel("Class rank")
    ax.set_ylabel("# proteins")
    ax.axhline(y=np.mean(sizes), color="red", linestyle="--", alpha=0.5, label=f"mean={np.mean(sizes):.0f}")
    ax.legend(fontsize=8)

fig.suptitle(f"HOG {ROOTHOG} — Sub-HOG class sizes by level", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Depth distribution within HOG 801468

Understanding why coverage drops at deeper levels.

In [ ]:
h = get_hog_proteins(ROOTHOG)
depth_counts = h["depth"].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Depth histogram
ax1.bar(depth_counts.index, depth_counts.values, color="teal", alpha=0.7)
ax1.set_xlabel("HOG depth")
ax1.set_ylabel("# proteins")
ax1.set_title(f"HOG {ROOTHOG}: depth distribution ({len(h)} proteins)")

# Cumulative from the deep end
cumsum = depth_counts.sort_index(ascending=False).cumsum()
ax2.plot(cumsum.index, cumsum.values, "o-", color="teal")
ax2.set_xlabel("Minimum depth")
ax2.set_ylabel("# proteins with depth >= X")
ax2.set_title("Cumulative proteins by minimum depth")
ax2.axhline(y=200, color="red", linestyle="--", alpha=0.5, label="n=200")
ax2.legend()
ax2.invert_xaxis()

plt.tight_layout()
plt.show()

# Print exact numbers
print("Proteins with depth >= N:")
for d in range(0, 8):
    n = (h["depth"] >= d).sum()
    print(f"  depth >= {d}: {n:,}")

## Query any root HOG

Change `QUERY_HOG` and `QUERY_LEVEL` to explore other HOGs interactively.

In [ ]:
QUERY_HOG = 801468    # <-- change me
QUERY_LEVEL = 3       # <-- change me
MIN_CLASS = 5         # <-- minimum proteins per class
MIN_DEPTH = 0         # <-- set >0 to restrict to deep proteins only

stats = hog_level_stats(QUERY_HOG, QUERY_LEVEL, min_class_size=MIN_CLASS, min_depth=MIN_DEPTH)
h = get_hog_proteins(QUERY_HOG, min_depth=MIN_DEPTH)

print(f"HOG {QUERY_HOG} at level {QUERY_LEVEL}"
      + (f" (depth >= {MIN_DEPTH})" if MIN_DEPTH else ""))
print(f"Total proteins: {len(h)}")
print(f"Classes with >= {MIN_CLASS} proteins: {len(stats)}")
print(f"Coverage: {stats['count'].sum()}/{len(h)} ({100*stats['count'].sum()/len(h):.1f}%)")
print()
display(stats)

## UMAP visualization colored by sub-HOG

Load embeddings for HOG 801468, run UMAP, color by sub-HOG class at the chosen level.

In [ ]:
from umap import UMAP

# --- Config ---
VIZ_MODEL = "650M"      # which model's embeddings to use
VIZ_LEVEL = 3            # sub-HOG level for coloring
VIZ_MIN_CLASS = 5        # min proteins per class to get a color
VIZ_SEED = 42
# --------------

h = get_hog_proteins(ROOTHOG)
h["sub_hog"] = h["hog_id"].apply(lambda x: trunc_hog(x, VIZ_LEVEL))

# Class labels
counts = h["sub_hog"].value_counts()
big_classes = set(counts[counts >= VIZ_MIN_CLASS].index)
h["label"] = h["sub_hog"].where(h["sub_hog"].isin(big_classes), other="Other")
h["short_label"] = h["label"].apply(lambda x: x.split(".")[-1] if x != "Other" else "Other")

# Load embeddings
entry_ids, X = load_embeddings(VIZ_MODEL, set(h["EntryID"]))
# Align
idx_map = {eid: i for i, eid in enumerate(entry_ids)}
aligned = h[h["EntryID"].isin(idx_map)].copy()
X_aligned = X[[idx_map[eid] for eid in aligned["EntryID"]]]

print(f"Loaded {VIZ_MODEL} embeddings: {X_aligned.shape}")
print(f"Running UMAP...")

umap = UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=VIZ_SEED, verbose=False)
coords = umap.fit_transform(X_aligned)
aligned["umap_x"] = coords[:, 0]
aligned["umap_y"] = coords[:, 1]
print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

# Background: "Other" proteins
other = aligned[aligned["label"] == "Other"]
ax.scatter(other["umap_x"], other["umap_y"], c="lightgray", alpha=0.3, s=10,
           label=f"Other (n={len(other)})", zorder=1)

# Colored classes
colors = plt.cm.tab20.colors + plt.cm.tab20b.colors
label_order = (aligned[aligned["label"] != "Other"]["short_label"]
               .value_counts().index.tolist())

for i, lbl in enumerate(label_order):
    grp = aligned[aligned["short_label"] == lbl]
    ax.scatter(grp["umap_x"], grp["umap_y"], c=[colors[i % len(colors)]],
               alpha=0.8, s=25, label=f"{lbl} (n={len(grp)})", zorder=2)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(f"HOG {ROOTHOG} — {VIZ_MODEL} embeddings\n"
             f"colored by level-{VIZ_LEVEL} sub-HOG (min {VIZ_MIN_CLASS} proteins)")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=7,
          title=f"Sub-HOG (L{VIZ_LEVEL})")
plt.tight_layout()
plt.show()

## Silhouette by sub-HOG class

Compute per-class silhouette for the colored sub-HOGs only (skip "Other").

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

labeled = aligned[aligned["label"] != "Other"].copy()
labeled_ids = set(labeled["EntryID"])

# Get aligned embeddings for labeled proteins only
labeled_X = X_aligned[[i for i, eid in enumerate(aligned["EntryID"].values) 
                         if eid in labeled_ids]]

# Encode labels as integers
label_map = {l: i for i, l in enumerate(labeled["short_label"].unique())}
y = labeled["short_label"].map(label_map).values

if len(np.unique(y)) >= 2:
    # Overall silhouette
    sil_orig = silhouette_score(labeled_X, y, metric="cosine")
    
    # UMAP space
    umap_coords = labeled[["umap_x", "umap_y"]].values
    sil_umap = silhouette_score(umap_coords, y, metric="euclidean")
    
    # Per-class
    sil_per = silhouette_samples(labeled_X, y, metric="cosine")
    labeled["sil"] = sil_per
    
    per_class = (labeled.groupby("short_label")["sil"]
                 .agg(["mean", "std", "count"])
                 .sort_values("mean", ascending=False))
    
    print(f"Overall silhouette ({VIZ_MODEL}, cosine): {sil_orig:.4f}")
    print(f"Overall silhouette (UMAP, euclidean):     {sil_umap:.4f}")
    print(f"\nPer-class ({len(per_class)} classes, {len(labeled)} proteins):")
    display(per_class.round(4))
else:
    print("Need at least 2 classes for silhouette.")

## Multi-model comparison on sub-HOG classes

Compare 150M, 650M, 3B silhouette on the same sub-HOG labels.

In [ ]:
CMP_LEVEL = 3  # <-- change to compare at different levels

h = get_hog_proteins(ROOTHOG)
h["sub_hog"] = h["hog_id"].apply(lambda x: trunc_hog(x, CMP_LEVEL))
counts = h["sub_hog"].value_counts()
big = set(counts[counts >= 5].index)
labeled = h[h["sub_hog"].isin(big)].copy()
label_map = {l: i for i, l in enumerate(labeled["sub_hog"].unique())}
y = labeled["sub_hog"].map(label_map).values

print(f"HOG {ROOTHOG}, Level {CMP_LEVEL}: {len(label_map)} classes, {len(labeled)} proteins\n")
print(f"{'Model':<8} {'Dim':>6} {'Sil (cosine)':>14}")
print("-" * 30)

for model_name in ["150M", "650M", "3B"]:
    eids, X = load_embeddings(model_name, set(labeled["EntryID"]))
    idx = {eid: i for i, eid in enumerate(eids)}
    X_sub = X[[idx[eid] for eid in labeled["EntryID"].values if eid in idx]]
    
    if len(X_sub) == len(y):
        s = silhouette_score(X_sub, y, metric="cosine")
        print(f"{model_name:<8} {X_sub.shape[1]:>6} {s:>14.4f}")
    else:
        print(f"{model_name:<8} alignment mismatch")

## HOG hierarchy tree (top levels)

Visualize the branching structure of HOG 801468 as a treemap.

In [ ]:
h = get_hog_proteins(ROOTHOG)

# Build a sunburst-style view: for each level, show the top N sub-HOGs
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, level in zip(axes, [1, 2, 3]):
    labels = h["hog_id"].apply(lambda x: trunc_hog(x, level))
    counts = labels.value_counts()
    top = counts.head(15)
    other = counts.iloc[15:].sum() if len(counts) > 15 else 0
    
    names = [c.split(".")[-1] for c in top.index]
    sizes = list(top.values)
    if other > 0:
        names.append(f"other ({len(counts)-15})")
        sizes.append(other)
    
    wedges, texts, autotexts = ax.pie(
        sizes, labels=None, autopct=lambda p: f"{p:.0f}%" if p > 3 else "",
        startangle=90, pctdistance=0.8,
        colors=plt.cm.tab20.colors[:len(sizes)])
    ax.set_title(f"Level {level}\n({len(counts)} total sub-HOGs)")
    
    # Only label big slices
    ax.legend(names, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=6)

fig.suptitle(f"HOG {ROOTHOG} — sub-HOG distribution by level", fontsize=13)
plt.tight_layout()
plt.show()

## Species composition per sub-HOG

Which organisms make up each sub-HOG class?

In [ ]:
SPECIES_LEVEL = 3  # <-- change me

h = get_hog_proteins(ROOTHOG)
h["sub_hog"] = h["hog_id"].apply(lambda x: trunc_hog(x, SPECIES_LEVEL))
h["short_label"] = h["sub_hog"].apply(lambda x: x.split(".")[-1])
h["species"] = h["scientific_name"].apply(
    lambda x: f"{x.split()[0][0]}. {x.split()[1]}" if isinstance(x, str) and len(x.split()) >= 2 else str(x)[:15])

counts = h["sub_hog"].value_counts()
top_classes = counts[counts >= 5].head(12).index

ct = pd.crosstab(h[h["sub_hog"].isin(top_classes)]["short_label"],
                 h[h["sub_hog"].isin(top_classes)]["species"])
# Keep only species with > 0 total
ct = ct.loc[:, ct.sum() > 0]
# Sort by total
ct = ct[ct.sum().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title(f"HOG {ROOTHOG} Level-{SPECIES_LEVEL} — species per sub-HOG")
ax.set_xlabel("Species")
ax.set_ylabel("Sub-HOG")
plt.tight_layout()
plt.show()